# Crypto-core trailing returns (BTC / ETH / SOL)

Reference experiment for the B-054/B-055 framework. Demonstrates the full
shape: seed -> pooled session -> `read_sql_df` -> transform -> `write_manifest`.
See `experiment.md` for the writeup.


## Setup — seed, session, manifest


In [ ]:
from pathlib import Path

from genkei.common.notebook import get_session, set_seeds, write_manifest

ASSETS = ["BTC-USD", "ETH-USD", "SOL-USD"]
WINDOW_DAYS = 30

seed = set_seeds(20260701)
session = get_session()
seed

## Query — latest close vs the close ~30 days earlier, per product


In [ ]:
SQL = """
    WITH latest AS (
        SELECT DISTINCT ON (product)
               product, close AS close_now, ts AS ts_now,
               ingest_run_id AS latest_ingest_run_id
        FROM coinbase.candles WHERE product = ANY(%s)
        ORDER BY product, ts DESC
    ),
    prior AS (
        SELECT DISTINCT ON (c.product)
               c.product, c.close AS close_prior, c.ts AS ts_prior,
               c.ingest_run_id AS prior_ingest_run_id
        FROM coinbase.candles c
        JOIN latest l USING (product)
        WHERE c.ts <= l.ts_now - make_interval(days => %s)
        ORDER BY c.product, c.ts DESC
    )
    SELECT l.product, l.close_now, p.close_prior,
           l.latest_ingest_run_id, p.prior_ingest_run_id,
           l.close_now / p.close_prior - 1 AS return_30d
    FROM latest l JOIN prior p USING (product)
    ORDER BY return_30d DESC
"""

df = session.read_sql_df(SQL, [ASSETS, WINDOW_DAYS])
df

## Result — ranked by trailing return

**Horizon tag:** crypto:core:years

Lowest-beta anchor (BTC) is expected to show the shallowest drawdown.


In [ ]:
df.assign(return_30d_pct=(df["return_30d"] * 100).round(2))[
    ["product", "close_now", "close_prior", "return_30d_pct"]
]

## Pin the snapshot


In [ ]:
write_manifest(
    Path("manifest.json"),
    seed=seed,
    config={"window_days": WINDOW_DAYS, "assets": ASSETS},
    data=df,
    sources=["coinbase"],
)
session.close()